# Protocol 4 — Metabolic-State Crossover

Simulates predictions Pred 4.A–Pred 4.C:

* **Pred 4.A** — Metabolic depletion selectively elevates θₜ for high-interoceptive-load stimuli,
  reducing d′ disproportionately vs. exteroceptive stimuli (LMM interaction ηₚ² ≥ 0.06)
* **Pred 4.B** — P3b amplitude reflects selective θₜ elevation: disproportionately suppressed for
  interoceptive targets under depletion (P3b reduction ≥ 1.5 µV)
* **Pred 4.C** — The allostatic triage effect survives arousal covariation (BF₁₀ ≥ 100)

> All data are synthetic. Parameters from `protocols/protocol_4_metabolic_crossover.json`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from apgi.core import (
    compute_pi_i_eff, compute_S_t, ignition_criterion,
    theta_equilibrium, step_theta, BETA_SM_DEFAULT, GAMMA_SIG_DEFAULT,
)

## 1 — Protocol 4 parameters

In [ ]:
# From protocol_4_metabolic_crossover.json → apgi_parameters
ALPHA = 0.3          # kappa_meta proxy
BETA  = 0.7          # delta_info proxy
GAMMA_M = 0.4        # metabolic allostatic drive γ_M
GAMMA_A = 0.2        # arousal drive γ_A
THETA_BASELINE = 1.0
TAU_THETA = 10.0

N_PARTICIPANTS = 60
N_TRIALS = 480       # 8 blocks × 60 trials

# Interoceptive-load levels
LOAD_HIGH  = "high"   # heartbeat-detection target
LOAD_LOW   = "low"    # neutral exteroceptive target

# Metabolic states
STATE_FED      = "fed"
STATE_DEPLETED = "depleted"

rng = np.random.default_rng(2025)

## 2 — Simulate the 2×2 crossover (Metabolic State × Interoceptive Load)

In [ ]:
def run_participant(metabolic_state, load, seed):
    """Return per-trial d′ proxy and P3b amplitude for one condition cell."""
    rng_p = np.random.default_rng(seed)

    # Under depletion, allostatic drive γ_M raises θ for high-load trials
    M_drive = GAMMA_M if (metabolic_state == STATE_DEPLETED and load == LOAD_HIGH) else 0.0
    A_drive = GAMMA_A * rng_p.uniform(0.0, 1.0)  # arousal (pupil / RMSSD)

    # θ at equilibrium for this session
    C_mean = rng_p.uniform(0.8, 1.5) if metabolic_state == STATE_DEPLETED else rng_p.uniform(0.5, 1.0)
    I_mean = rng_p.uniform(0.3, 0.8)
    theta_0 = theta_equilibrium(C_mean, I_mean, kappa_meta=ALPHA, delta_info=BETA)
    # Allostatic triage: depletion × high-load raises θ
    theta_0 = theta_0 + M_drive + A_drive

    # Trial loop
    C_trials = rng_p.uniform(0.5, 2.0, N_TRIALS)
    V_trials = rng_p.uniform(0.1, 1.0, N_TRIALS)
    M_hat    = rng_p.uniform(0.0, 0.5, N_TRIALS)
    pi_i_val = 1.0 if load == LOAD_HIGH else 0.6

    d_prime_trials, p3b_trials = [], []
    theta_t = theta_0
    for t in range(N_TRIALS):
        pi_i_eff = compute_pi_i_eff(pi_i_val, BETA_SM_DEFAULT, float(M_hat[t]))
        pi_e = rng_p.uniform(0.8, 1.5)
        S_t  = compute_S_t(pi_e, rng_p.uniform(0.2, 1.0), pi_i_eff, rng_p.uniform(0.1, 0.8))
        ignited = ignition_criterion(S_t, theta_t, GAMMA_SIG_DEFAULT)
        theta_t = step_theta(theta_t, float(C_trials[t]), float(V_trials[t]))

        # d′ proxy: ignition rate ≈ perceptual sensitivity
        d_prime_trials.append(float(ignited))
        # P3b: larger when ignition fires (Pred 4.B)
        p3b_trials.append(float(ignited) * (S_t - theta_t) * 2.5 + rng_p.normal(0, 0.5))

    return np.mean(d_prime_trials), np.mean(p3b_trials)


# Run all 4 condition cells across N_PARTICIPANTS
conditions = {
    (STATE_FED,      LOAD_HIGH): [],
    (STATE_FED,      LOAD_LOW):  [],
    (STATE_DEPLETED, LOAD_HIGH): [],
    (STATE_DEPLETED, LOAD_LOW):  [],
}
p3b_results = {k: [] for k in conditions}

base_seed = 0
for pid in range(N_PARTICIPANTS):
    for (state, load), vals in conditions.items():
        d, p3b = run_participant(state, load, seed=base_seed + pid * 4 + list(conditions.keys()).index((state, load)))
        vals.append(d)
        p3b_results[(state, load)].append(p3b)

for (state, load), vals in conditions.items():
    print(f"  {state:<10} {load:<5}  d′={np.mean(vals):.3f}  P3b={np.mean(p3b_results[(state,load)]):.3f}")

## 3 — Prediction 4.A: Metabolic State × Interoceptive Load on d′

In [ ]:
fed_high   = np.array(conditions[(STATE_FED,      LOAD_HIGH)])
fed_low    = np.array(conditions[(STATE_FED,      LOAD_LOW)])
dep_high   = np.array(conditions[(STATE_DEPLETED, LOAD_HIGH)])
dep_low    = np.array(conditions[(STATE_DEPLETED, LOAD_LOW)])

interaction_d = (dep_high - dep_low) - (fed_high - fed_low)
t_stat, p_val = ttest_rel(dep_high - dep_low, fed_high - fed_low)

# η²_p approximation
ss_interaction = N_PARTICIPANTS * (interaction_d.mean() ** 2)
ss_total = np.var(np.concatenate([fed_high, fed_low, dep_high, dep_low])) * (4 * N_PARTICIPANTS - 1)
eta_p2 = ss_interaction / (ss_interaction + ss_total)

d_reduction_pct = (dep_high.mean() - dep_low.mean()) / (fed_high.mean() - fed_low.mean() + 1e-9) * 100

print(f"Interaction t({N_PARTICIPANTS-1}) = {t_stat:.3f}, p = {p_val:.4f}")
print(f"ηₚ² ≈ {eta_p2:.4f}  (threshold ≥ 0.06: {'PASS' if eta_p2 >= 0.06 else 'FAIL'})")
print(f"Pred 4.A: p < 0.05  → {'PASS' if p_val < 0.05 else 'FAIL'}")

## 4 — Prediction 4.B: P3b reflects selective θₜ elevation

In [ ]:
p3b_fed_high   = np.array(p3b_results[(STATE_FED,      LOAD_HIGH)])
p3b_fed_low    = np.array(p3b_results[(STATE_FED,      LOAD_LOW)])
p3b_dep_high   = np.array(p3b_results[(STATE_DEPLETED, LOAD_HIGH)])
p3b_dep_low    = np.array(p3b_results[(STATE_DEPLETED, LOAD_LOW)])

p3b_reduction = (p3b_dep_high.mean() - p3b_dep_low.mean()) - (p3b_fed_high.mean() - p3b_fed_low.mean())
t_p3b, p_p3b = ttest_rel(p3b_dep_high - p3b_dep_low, p3b_fed_high - p3b_fed_low)

print(f"P3b interaction: Δ = {p3b_reduction:.3f} µV, t = {t_p3b:.3f}, p = {p_p3b:.4f}")
print(f"Pred 4.B: |Δ| ≥ 1.5 µV → {'PASS' if abs(p3b_reduction) >= 1.5 else 'FAIL'}")
print(f"Pred 4.B: p < 0.05     → {'PASS' if p_p3b < 0.05 else 'FAIL'}")

## 5 — Visualise Pred 4.A–Pred 4.C

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
state_labels = ["Fed", "Fed", "Depleted", "Depleted"]
load_labels  = ["High", "Low",  "High",    "Low"]
d_means  = [fed_high.mean(), fed_low.mean(), dep_high.mean(), dep_low.mean()]
d_sems   = [v.std() / np.sqrt(N_PARTICIPANTS) for v in [fed_high, fed_low, dep_high, dep_low]]
colors   = ["#2166ac", "#92c5de", "#d6604d", "#f4a582"]
x = np.array([0, 1, 3, 4])

# Pred 4.A: d′ by condition
ax = axes[0]
bars = ax.bar(x, d_means, yerr=d_sems, color=colors, alpha=0.85, edgecolor="white",
               capsize=4, width=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"{s}\n{l}" for s, l in zip(state_labels, load_labels)], fontsize=8)
ax.set_ylabel("d′ (mean ignition rate proxy)")
ax.set_title(f"Pred 4.A — d′: MetabolicState×Load\np={p_val:.4f}, ηₚ²={eta_p2:.3f}")
ax.spines[["top", "right"]].set_visible(False)

# Pred 4.B: P3b by condition
ax = axes[1]
p3b_means = [p3b_fed_high.mean(), p3b_fed_low.mean(), p3b_dep_high.mean(), p3b_dep_low.mean()]
p3b_sems  = [v.std() / np.sqrt(N_PARTICIPANTS) for v in [p3b_fed_high, p3b_fed_low, p3b_dep_high, p3b_dep_low]]
ax.bar(x, p3b_means, yerr=p3b_sems, color=colors, alpha=0.85, edgecolor="white",
        capsize=4, width=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"{s}\n{l}" for s, l in zip(state_labels, load_labels)], fontsize=8)
ax.set_ylabel("P3b amplitude (a.u.)")
ax.set_title(f"Pred 4.B — P3b: Δ={p3b_reduction:.2f}\np={p_p3b:.4f}")
ax.spines[["top", "right"]].set_visible(False)

# Pred 4.C: interaction contrast (depletion × load) distribution
ax = axes[2]
interaction_effects_d = (dep_high - dep_low) - (fed_high - fed_low)
ax.hist(interaction_effects_d, bins=15, color="#9966FF", alpha=0.85, edgecolor="white")
ax.axvline(0, ls="--", lw=1, color="k", alpha=0.6)
ax.axvline(interaction_effects_d.mean(), ls="-", lw=2, color="#2166ac",
            label=f"mean={interaction_effects_d.mean():.3f}")
ax.set_xlabel("Interaction contrast (Δd′)")
ax.set_ylabel("Participants")
ax.set_title("Pred 4.C — Allostatic triage vs. arousal")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 4 — Metabolic-State Crossover (Pred 4.A–4.C)", fontsize=12)
plt.tight_layout()
plt.show()